# 02 — Gate 1: data spine + cross-dataset dedup audit
Hard go/no-go. Pull three role datasets, harmonise, check whether a transport gap would be real shift or an artefact of shared near-duplicate prompts.

direct+benign -> `deepset/prompt-injections` · over-defense -> `leolee99/NotInject` · indirect -> `microsoft/BIPIA` (best-effort).

Dedup = MinHash + LSH at Jaccard >= 0.8 (Gate AI protocol, arXiv 2606.02959; cite, not novel).

## Session bootstrap

In [1]:
# === SESSION BOOTSTRAP (run first, every session) ===
# A fresh Colab runtime forgets --global git config, so we re-set it here.
from google.colab import drive
drive.mount('/content/drive')
import os, sys
DRIVE_ROOT = '/content/drive/MyDrive/PICALIB_Research'
REPO_DIR   = os.path.join(DRIVE_ROOT, 'picalib-research')
!git config --global user.name  "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git config --global credential.helper "store --file={DRIVE_ROOT}/.git-credentials"
%cd {REPO_DIR}
sys.path.insert(0, 'src')          # so `import metrics` finds src/metrics.py
!git pull
print('Session ready - identity set, src/ on path, repo pulled.')

Mounted at /content/drive
/content/drive/MyDrive/PICALIB_Research/picalib-research
Already up to date.
Session ready - identity set, src/ on path, repo pulled.


## Dependencies

In [2]:
!pip install -q datasets datasketch pandas
print('deps installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 1.6 MB/s eta 0:00:00
deps installed


## Load + harmonise (never hard-fails on one source)

In [3]:
import importlib, data_loaders, dedup
importlib.reload(data_loaders); importlib.reload(dedup)
from data_loaders import load_all
import json, pandas as pd

df, report = load_all(include_bipia=True)
print('\nLoad report:'); print(json.dumps(report, indent=2))
print('\nsource / unit_type / label:')
print(df.groupby(['source','unit_type','label']).size())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/546 [00:00<?, ? examples/s]

...

data/NotInject_three-00000-of-00001.parq(…):   0%|          | 0.00/13.0k [00:00<?, ?B/s]

Generating NotInject_one split:   0%|          | 0/113 [00:00<?, ? examples/s]

Generating NotInject_two split:   0%|          | 0/113 [00:00<?, ? examples/s]

Generating NotInject_three split:   0%|          | 0/113 [00:00<?, ? examples/s]

[ok]   notinject  n=339
Cloning microsoft/BIPIA ...
BIPIA: found 4 candidate attack files.
   BIPIA/benchmark/code_attack_test.json
   BIPIA/benchmark/text_attack_train.json
   BIPIA/benchmark/code_attack_train.json
   BIPIA/benchmark/text_attack_test.json
BIPIA: parsed 250 attack instruction strings.
[ok]   bipia      n=250

Load report:
{
  "deepset": {
    "ok": true,
    "n": 662,
    "n_attack": 263,
    "n_benign": 399
  },
  "notinject": {
    "ok": true,
    "n": 339,
    "n_attack": 0,
    "n_benign": 339
  },
  "bipia": {
    "ok": true,
    "n": 250,
    "n_attack": 250,
    "n_benign": 0
  }
}

source / unit_type / label:
source     unit_type                label
bipia      instruction_in_document  1        250
deepset    standalone_prompt        0        399
                                    1        263
notinject  standalone_prompt        0        339
dtype: int64


## Detection-unit manifest (standalone vs in-document)

In [4]:
import os
os.makedirs('data', exist_ok=True); os.makedirs('reports', exist_ok=True)
df.to_csv('data/harmonised.csv', index=False)   # gitignored
manifest = df.groupby(['source','unit_type','label']).size().reset_index(name='n')
manifest.to_csv('reports/gate1_manifest.csv', index=False)
print(manifest.to_string(index=False))

   source               unit_type  label   n
    bipia instruction_in_document      1 250
  deepset       standalone_prompt      0 399
  deepset       standalone_prompt      1 263
notinject       standalone_prompt      0 339


## Cross-dataset near-duplicate audit (the go/no-go)

In [5]:
from dedup import dedup_audit, print_report
audit = dedup_audit(df, threshold=0.8, k=3)
print_report(audit)

Per-source counts: {'deepset': 662, 'notinject': 339, 'bipia': 250}

Cross-source near-duplicate overlap (fraction of ROW source with a dup in COLUMN source):
           bipia  deepset  notinject
bipia        0.0   0.0000        0.0
deepset      0.0   0.0438        0.0
notinject    0.0   0.0000        0.0

Example cross-source near-duplicate pairs:

VERDICT: GO — max cross-source overlap 0.0% (< 5%) — datasets are distinct enough.


## Save audit + verdict

In [6]:
audit['overlap_frac'].to_csv('reports/gate1_overlap_frac.csv')
v, reason = audit['verdict']
open('reports/gate1_verdict.txt','w').write(f'{v} - {reason}\n')
print('GATE 1:', v)
if v != 'GO':
    print('-> inspect example pairs above; swap a backup source before scaling up.')

GATE 1: GO


---
## Commit + push

In [7]:
!git add -A
!git commit -m "Gate 1: data spine + MinHash-LSH cross-dataset dedup audit"
!git push
print('Pushed. Confirm GitHub + Drive in sync.')

[main fbb8533] Gate 1: data spine + MinHash-LSH cross-dataset dedup audit
 6 files changed, 2331 insertions(+), 349 deletions(-)
 create mode 100644 data/harmonised.csv
 rewrite notebooks/01_metric_harness.ipynb (99%)
 rewrite notebooks/02_gate1_data_dedup.ipynb (100%)
 create mode 100644 reports/gate1_manifest.csv
 create mode 100644 reports/gate1_overlap_frac.csv
 create mode 100644 reports/gate1_verdict.txt
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (11/11), 85.72 KiB | 2.14 MiB/s, done.
Total 11 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/anasbiswas1/picalib-research.git
   c671166..fbb8533  main -> main
Pushed. Confirm GitHub + Drive in sync.


In [8]:
%cd {REPO_DIR}
open('.gitignore','w').write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.git-credentials\ndata/\n_bipia/\n")
!git rm --cached -r data
!git add .gitignore
!git commit -m "Stop tracking data/ (raw datasets); fix .gitignore"
!git push
print("done — data/ now untracked, .gitignore fixed")

/content/drive/MyDrive/PICALIB_Research/picalib-research
rm 'data/harmonised.csv'
[main 9a57640] Stop tracking data/ (raw datasets); fix .gitignore
 2 files changed, 2 insertions(+), 2319 deletions(-)
 delete mode 100644 data/harmonised.csv
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 374 bytes | 62.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/anasbiswas1/picalib-research.git
   fbb8533..9a57640  main -> main
done — data/ now untracked, .gitignore fixed
